# TimelyMT Research MVP: Kaggle Git-Clone Runner
Import only this notebook into Kaggle, enable a GPU accelerator and Internet, then run all cells in order. The notebook clones the public repository and stops after frozen TRAIN/DEV selection; it never executes the held-out evaluation split.

## PERSISTENT CHECKPOINTING
1. Create one **PRIVATE** Kaggle Dataset manually.
2. Set `CHECKPOINT_DATASET_REF` below, replacing only the owner if necessary.
3. Add Kaggle Secret `KAGGLE_API_TOKEN`.
4. Run All. The latest compatible checkpoint restores automatically.
5. New private Dataset versions are written after expensive TRAIN/DEV stages.

In [ ]:
from datetime import datetime, timezone

SESSION_STARTED_AT = datetime.now(timezone.utc)
REPO_URL = "https://github.com/MinhCYB/TimelyMT.git"
REPO_BRANCH = "main"
REPO_DIR = "/kaggle/working/TimelyMT"
SRC_DIR = "/kaggle/working/TimelyMT/src"

CHECKPOINT_DATASET_REF = "iteams24/timelymt-research-checkpoints"
RUN_EMERGENCY_CHECKPOINT = False
INFERENCE_BATCH_SIZE = 3

HF_CACHE_DIR = "/kaggle/temp/huggingface"

## CLONE REPOSITORY
Clone shallow `main` from GitHub. Rerunning this cell refreshes only the disposable Kaggle checkout while retaining valid ignored run artifacts in the current session. No path under `/kaggle/input` is read or modified.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

repo = Path(REPO_DIR)
expected_remote = REPO_URL.removesuffix(".git").rstrip("/").lower()

def run(command, *, cwd=None, env=None, capture_output=False):
    print("+", " ".join(map(str, command)), flush=True)
    return subprocess.run(
        list(map(str, command)), cwd=cwd, check=True, text=True,
        env=env, capture_output=capture_output,
    )

valid_checkout = False
if repo.exists() and (repo / ".git").is_dir():
    remote = run(["git", "remote", "get-url", "origin"], cwd=repo, capture_output=True).stdout.strip()
    valid_checkout = remote.removesuffix(".git").rstrip("/").lower() == expected_remote

if valid_checkout:
    run(["git", "fetch", "--depth", "1", "origin", REPO_BRANCH], cwd=repo)
    run(["git", "reset", "--hard", "FETCH_HEAD"], cwd=repo)
else:
    if repo.exists():
        print(f"Removing unexpected disposable path: {repo}", flush=True)
        shutil.rmtree(repo)
    run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, repo])

os.chdir(repo)
commit = run(["git", "rev-parse", "HEAD"], cwd=repo, capture_output=True).stdout.strip()
status = run(["git", "status", "--short"], cwd=repo, capture_output=True).stdout
print(f"Experiment commit: {commit}")
print("git status --short:")
print(status if status else "(clean)")
git_clone_passed = True

## INSTALL DEPENDENCIES / SOURCE-TREE IMPORT BOOTSTRAP
Use Kaggle's preinstalled CUDA-enabled PyTorch, install the remaining runtime dependencies, then make the cloned `src` tree authoritative for this kernel and every TimelyMT child process. No kernel restart or editable install is required.

In [ ]:
TRANSFORMERS_REQUIREMENT = "transformers>=4.57.6,<5.0.0"
hf_cache = Path(HF_CACHE_DIR)
hf_cache.mkdir(parents=True, exist_ok=True)
os.environ.update({
    "HF_HOME": HF_CACHE_DIR,
    "HF_HUB_CACHE": str(hf_cache / "hub"),
    "TOKENIZERS_PARALLELISM": "false",
})

# Install runtime dependencies explicitly so pip never replaces Kaggle's CUDA-enabled torch.
run([
    sys.executable, "-m", "pip", "install",
    "joblib", "sacrebleu", "scikit-learn", "sentencepiece", TRANSFORMERS_REQUIREMENT,
])
try:
    from importlib.metadata import version as package_version
    kaggle_cli_version = package_version("kaggle")
except Exception:
    run([sys.executable, "-m", "pip", "install", "kaggle>=2.2.0"])
    kaggle_cli_version = package_version("kaggle")

src_dir = Path(SRC_DIR).resolve()
if not src_dir.is_dir():
    raise RuntimeError(f"Cloned source directory is missing: {src_dir}")

# Make the clone authoritative even if this kernel previously imported an installed wheel.
sys.path[:] = [
    entry for entry in sys.path
    if Path(entry or os.curdir).resolve() != src_dir
]
sys.path.insert(0, str(src_dir))
import importlib
for module_name in [name for name in sys.modules if name == "timelymt" or name.startswith("timelymt.")]:
    del sys.modules[module_name]
importlib.invalidate_caches()
import timelymt
import timelymt.research.cli as research_cli

package_dir = (src_dir / "timelymt").resolve()
module_path = Path(timelymt.__file__).resolve()
cli_module_path = Path(research_cli.__file__).resolve()
project_root = Path(research_cli.ROOT).resolve()
split_path = (project_root / "data/splits/experimental.json").resolve()
if sys.path[0] != str(src_dir) or sum(Path(entry or os.curdir).resolve() == src_dir for entry in sys.path) != 1:
    raise RuntimeError(f"Source directory is not unique and first on sys.path: {sys.path[:5]}")
if not module_path.is_relative_to(package_dir):
    raise RuntimeError(f"TimelyMT did not load from cloned source: {module_path}")
if not cli_module_path.is_relative_to(package_dir) or project_root != repo.resolve():
    raise RuntimeError(f"Research CLI root mismatch: module={cli_module_path}, ROOT={project_root}")
if split_path != (repo / "data/splits/experimental.json").resolve() or not split_path.is_file():
    raise RuntimeError(f"Research CLI resolved the wrong Dataset v1 split: {split_path}")

existing_pythonpath = [
    entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep)
    if entry and Path(entry).resolve() != src_dir
]
child_env = os.environ.copy()
child_env["PYTHONPATH"] = os.pathsep.join([str(src_dir), *existing_pythonpath])
child_check = run(
    [
        sys.executable, "-c",
        "import timelymt; import timelymt.research.cli as c; "
        "print(timelymt.__file__); print(c.__file__); print(c.ROOT)",
    ],
    cwd=repo, env=child_env, capture_output=True,
).stdout.strip().splitlines()
if len(child_check) != 3:
    raise RuntimeError(f"Unexpected source-root child preflight output: {child_check}")
child_module_path, child_cli_path, child_root = map(lambda value: Path(value).resolve(), child_check)
if not child_module_path.is_relative_to(package_dir) or not child_cli_path.is_relative_to(package_dir) or child_root != repo.resolve():
    raise RuntimeError(f"Child process did not resolve cloned TimelyMT: {child_check}")
if (child_root / "data/splits/experimental.json").resolve() != split_path:
    raise RuntimeError(f"Child process resolved the wrong Dataset v1 root: {child_root}")

pip_version = run([sys.executable, "-m", "pip", "--version"], capture_output=True).stdout.strip()

print("PACKAGE / ROOT PREFLIGHT")
print("------------------------")
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
print(f"pip version: {pip_version}")
print(f"Repository root: {repo.resolve()}")
print(f"Source root: {src_dir}")
print(f"Git commit: {commit}")
print("TimelyMT import: OK")
print(f"TimelyMT module path: {module_path}")
print(f"Research CLI module path: {cli_module_path}")
print(f"Resolved project ROOT: {project_root}")
print(f"Resolved Dataset v1 split: {split_path}")
print(f"Child TimelyMT module path: {child_module_path}")
print(f"Child Research CLI module path: {child_cli_path}")
print(f"Child project ROOT: {child_root}")
source_tree_import_passed = True
project_root_passed = True
package_preflight_passed = True

## ENVIRONMENT PRECHECK
Verify every required runtime import and Kaggle's CUDA accelerator before validating Dataset v1. The frozen PyTorch installation is never reinstalled.
<!-- Compatibility heading: ## MODEL/CACHE SETUP -->

In [ ]:
if not package_preflight_passed:
    raise RuntimeError("Package preflight did not pass")

try:
    import joblib
    import sacrebleu
    import sentencepiece
    import sklearn
    import torch
    import transformers
except ImportError as error:
    raise RuntimeError(f"Required runtime package is unavailable: {error.name}") from error

transformers_major_minor = tuple(map(int, transformers.__version__.split(".")[:2]))
if not ((4, 57) <= transformers_major_minor < (5, 0)):
    raise RuntimeError(f"Unsupported transformers version: {transformers.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a Kaggle GPU accelerator before continuing.")
gpu_capability = torch.cuda.get_device_capability(0)
gpu_arch = f"sm_{gpu_capability[0]}{gpu_capability[1]}"
supported_arches = torch.cuda.get_arch_list()
if gpu_arch not in supported_arches:
    raise RuntimeError(f"GPU architecture {gpu_arch} is unsupported by this PyTorch build: {supported_arches}")
torch.empty(1, device="cuda").add_(1)
torch.cuda.synchronize()

print(f"joblib: {joblib.__version__}")
print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"sentencepiece: {getattr(sentencepiece, '__version__', 'unknown')}")
print(f"sklearn: {sklearn.__version__}")
print(f"sacrebleu: {sacrebleu.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU architecture: {gpu_arch}")
print(f"Inference batch size: {INFERENCE_BATCH_SIZE}")
environment_preflight_passed = True
cuda_preflight_passed = True

def ensure_train_dev_only(args):
    if any(str(arg).lower() == "test" for arg in args):
        raise RuntimeError("TEST execution is forbidden in this TRAIN/DEV notebook")

def cli(*args):
    ensure_train_dev_only(args)
    command = [sys.executable, "-u", "-m", "timelymt.research.cli", *args]
    run(command, cwd=repo, env=child_env)

def require_file(relative_path):
    path = repo / relative_path
    if not path.is_file():
        raise RuntimeError(f"Missing required artifact: {path}")
    return path

try:
    ensure_train_dev_only(("evaluate", "--split", "test"))
except RuntimeError:
    test_safeguard_passed = True
else:
    raise RuntimeError("TEST safeguard self-check failed")

## PRECHECK
Validate the cloned frozen Dataset v1 and experimental split before downloading EnViT5. This cell never reacquires TED data or rebuilds M0.

In [ ]:
if not (package_preflight_passed and environment_preflight_passed):
    raise RuntimeError("Package and environment preflights must pass before Dataset v1 validation")

import json
from timelymt.data.canonical.core import load_canonical_talk
from timelymt.data.manifest.core import validate_split_manifest
from timelymt.data.pipeline.qa import stable_checksum, validate_dataset
from timelymt.data.translation_artifacts import runtime_talk_from_canonical, stable_fingerprint
from timelymt.research.cli import DATASET_CHECKSUM, SPLIT_CHECKSUM, _manifests

required_paths = [
    "configs/translator/envit5.json",
    "configs/experiments/research-mvp.json",
    "data/manifests/streaming-dataset.json",
    "data/manifests/timelymt-streaming-dataset-v1.json",
    "data/splits/experimental.json",
    "schemas/streaming-talk.schema.json",
    "src/timelymt/research/cli.py",
]
for required in required_paths:
    require_file(required)

manifest = json.loads(require_file("data/manifests/streaming-dataset.json").read_text(encoding="utf-8"))
snapshot = json.loads(require_file("data/manifests/timelymt-streaming-dataset-v1.json").read_text(encoding="utf-8"))
split = json.loads(require_file("data/splits/experimental.json").read_text(encoding="utf-8"))
expected_ids = {row["talk_id"] for row in manifest["talks"]}
canonical_paths = {path.parent.name: path for path in (repo / "data/streaming/processed").glob("*/streaming-talk.json")}
if len(expected_ids) != 17 or set(canonical_paths) != expected_ids:
    raise RuntimeError({
        "expected_count": len(expected_ids),
        "actual_count": len(canonical_paths),
        "missing": sorted(expected_ids - set(canonical_paths)),
        "extra": sorted(set(canonical_paths) - expected_ids),
    })

dataset_result = validate_dataset(manifest, project_root=repo)
validate_split_manifest(split, manifest)
for row in manifest["talks"]:
    document = load_canonical_talk(repo / row["canonical_path"])
    runtime_talk_from_canonical(
        document, split_manifest=split,
        observed_through_token_index=len(document["stream"]["tokens"]) - 1,
    )
if not (
    dataset_result["manifest_checksum"] == snapshot["manifest_checksum"] == DATASET_CHECKSUM
    and stable_checksum(split) == stable_fingerprint(split) == snapshot["split_manifest_checksum"] == SPLIT_CHECKSUM
):
    raise RuntimeError("Frozen Dataset v1 or split identity changed")
_manifests()
dataset_preflight_passed = True
split_identity_passed = True
print(json.dumps({
    "dataset": dataset_result,
    "snapshot_version": snapshot["snapshot_version"],
    "split_checksum": stable_fingerprint(split),
    "split_counts": {name: len(ids) for name, ids in split["splits"].items()},
    "canonical_runtime_talks_validated": len(canonical_paths),
}, indent=2, sort_keys=True))

## CHECKPOINT AUTHENTICATION + RESTORE TOOLS
Authenticate through Kaggle Secrets without writing `kaggle.json`. The preflight reads Dataset status only; it never uploads. Checkpoint archives contain only generated TRAIN/DEV research artifacts and non-secret metadata.

In [ ]:
import gzip
import hashlib
import io
import re
import tarfile
import tempfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

CHECKPOINT_ARCHIVE = Path("/kaggle/working/timelymt-checkpoint.tar.gz")
CHECKPOINT_UPLOAD_DIR = Path("/kaggle/working/timelymt-checkpoint-upload")
CHECKPOINT_SCHEMA_VERSION = "1.0.0"
ALLOWED_CHECKPOINT_ROOTS = (
    PurePosixPath("data/policy"),
    PurePosixPath("checkpoints/policy"),
    PurePosixPath("outputs/experiments/research-mvp"),
)
CHECKPOINT_STAGE_ORDER = {
    "train-supervision-complete": 1,
    "dev-supervision-and-policies-complete": 2,
    "dev-baselines-complete": 3,
    "dev-frozen-complete": 4,
    "manual-emergency": 0,
}
CHECKPOINT_STAGE_REQUIREMENTS = {
    "train-supervision-complete": {"train-supervision-complete"},
    "dev-supervision-and-policies-complete": {"train-supervision-complete", "dev-supervision-and-policies-complete"},
    "dev-baselines-complete": {"train-supervision-complete", "dev-supervision-and-policies-complete", "dev-baselines-complete"},
    "dev-frozen-complete": set(CHECKPOINT_STAGE_ORDER) - {"manual-emergency"},
    "manual-emergency": set(),
}

def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")

def elapsed_session():
    elapsed = datetime.now(timezone.utc) - SESSION_STARTED_AT
    print(f"Session elapsed: {elapsed.total_seconds() / 3600:.2f} hours")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def checkpoint_dataset_ref_valid():
    owner, separator, slug = CHECKPOINT_DATASET_REF.partition("/")
    placeholders = {"owner", "your-owner", "your_username", "username", "placeholder"}
    return bool(separator and owner and slug and owner.lower() not in placeholders and not re.search(r"[<>{}]", CHECKPOINT_DATASET_REF))

def kaggle_command(*args, capture_output=False):
    return run([sys.executable, "-m", "kaggle", *args], capture_output=capture_output)

def authenticate_kaggle():
    if not checkpoint_dataset_ref_valid():
        raise RuntimeError(f"Set CHECKPOINT_DATASET_REF to the existing private Dataset before checkpoint persistence: {CHECKPOINT_DATASET_REF!r}")
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    except Exception as error:
        raise RuntimeError("Kaggle Secret KAGGLE_API_TOKEN is required; persistent research execution is blocked") from error
    if not token:
        raise RuntimeError("Kaggle Secret KAGGLE_API_TOKEN is empty; persistent research execution is blocked")
    os.environ["KAGGLE_API_TOKEN"] = token
    del token
    result = kaggle_command("datasets", "status", CHECKPOINT_DATASET_REF, "--format", "json", capture_output=True)
    if not result.stdout.strip():
        raise RuntimeError("Authenticated Kaggle Dataset status preflight returned no status")
    print(f"Kaggle CLI version: {kaggle_cli_version}")
    print(f"Checkpoint Dataset ref: {CHECKPOINT_DATASET_REF}")
    print("Authentication status: OK (private Dataset status readable)")
    return json.loads(result.stdout)

def load_json_if_valid(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except (OSError, UnicodeError, json.JSONDecodeError):
        return None

EXPECTED_TALKS = {name: tuple(split["splits"][name]) for name in ("train", "dev")}
TRANSLATOR_CONFIG_FINGERPRINT = stable_fingerprint(json.loads(require_file("configs/translator/envit5.json").read_text(encoding="utf-8")))

def supervision_status(kind, split_name):
    directory = repo / "data/policy" / ("pseudo_labels" if kind == "pseudo" else "mu_zhang2020") / split_name
    validator = research_cli._validate_pseudo_talk_file if kind == "pseudo" else research_cli._validate_mu_talk_file
    valid = []
    for talk_id in EXPECTED_TALKS[split_name]:
        talk_path = directory / f"{talk_id}.jsonl"
        if not talk_path.is_file():
            continue
        try:
            validator(talk_path, talk_id, split_name)
        except RuntimeError:
            continue
        valid.append(talk_id)
    manifest_doc = load_json_if_valid(directory / "manifest.json") or {}
    full = (
        set(valid) == set(EXPECTED_TALKS[split_name])
        and manifest_doc.get("artifact_status") == "full"
        and manifest_doc.get("publishable") is True
        and manifest_doc.get("split") == split_name
        and manifest_doc.get("dataset_checksum") == DATASET_CHECKSUM
        and manifest_doc.get("split_checksum") == SPLIT_CHECKSUM
        and set(manifest_doc.get("talk_ids", ())) == set(EXPECTED_TALKS[split_name])
    )
    return {"valid": len(valid), "expected": len(EXPECTED_TALKS[split_name]), "full": full}

def policy_checkpoint_valid(name):
    joblib_path = repo / "checkpoints/policy" / f"{name}.joblib"
    metadata = load_json_if_valid(repo / "checkpoints/policy" / f"{name}.metadata.json") or {}
    expected_identity = metadata.get("variant") == name if name in {"P0", "P1", "P2"} else metadata.get("strategy") == "mu_zhang2020"
    if not (joblib_path.is_file() and metadata.get("artifact_status") == "full" and metadata.get("publishable") is True and expected_identity):
        return False
    if metadata.get("checkpoint_sha256") != sha256_file(joblib_path):
        return False
    try:
        joblib.load(joblib_path)
    except Exception:
        return False
    return True

def prediction_strategy_valid(strategy):
    directory = repo / "outputs/experiments/research-mvp/predictions/dev" / strategy
    records = []
    for path in sorted(directory.glob("*.json")):
        document = load_json_if_valid(path)
        if document is None:
            return False
        records.append(document)
    return bool(records) and {row.get("talk_id") for row in records} == set(EXPECTED_TALKS["dev"]) and all(
        row.get("split") == "dev" and row.get("strategy") == strategy
        and row.get("artifact_status") == "full" and row.get("publishable") is True
        and isinstance(row.get("commits"), list) and isinstance(row.get("prediction"), str)
        for row in records
    )

FIXED_STRATEGIES = ["fixed_n_4", "fixed_n_8", "fixed_n_12", "fixed_time_1600", "fixed_time_3200", "fixed_time_4800"]
STYLE_STRATEGIES = ["local_agreement_style_k2", "local_agreement_style_k3"]
BASELINE_STRATEGIES = FIXED_STRATEGIES + STYLE_STRATEGIES + ["local_agreement_la2", "mu_zhang2020"]
LEARNED_STRATEGIES = [f"learned_{variant}_{threshold:.2f}" for variant in ("P0", "P1", "P2") for threshold in (0.30, 0.40, 0.50, 0.60, 0.70)]

def metrics_valid():
    document = load_json_if_valid(repo / "outputs/experiments/research-mvp/metrics/dev/all.json")
    expected = set(BASELINE_STRATEGIES + LEARNED_STRATEGIES)
    return isinstance(document, dict) and set(document) == expected and all(
        row.get("artifact_status") == "full" and row.get("publishable") is True
        for row in document.values()
    )

def selection_valid():
    document = load_json_if_valid(repo / "outputs/experiments/research-mvp/dev-selection.json") or {}
    return document.get("selected_strategy") in LEARNED_STRATEGIES and document.get("selected_variant") in {"P0", "P1", "P2"}

def frozen_config_valid():
    document = load_json_if_valid(repo / "outputs/experiments/research-mvp/frozen-eval-config.json") or {}
    required = {"dataset_checksum", "split_checksum", "translator", "trained_checkpoint_hashes", "selected_learned_variant", "selected_learned_threshold", "baseline_config", "evaluation_metric_config"}
    return required.issubset(document) and document.get("dataset_checksum") == DATASET_CHECKSUM and document.get("split_checksum") == SPLIT_CHECKSUM

def artifact_state():
    state = {}
    for split_name in ("train", "dev"):
        state[f"pseudo_{split_name}"] = supervision_status("pseudo", split_name)
        state[f"mu_{split_name}"] = supervision_status("mu", split_name)
    state["policies"] = {name: policy_checkpoint_valid(name) for name in ("P0", "P1", "P2", "mu_zhang2020")}
    state["baseline_predictions"] = {name: prediction_strategy_valid(name) for name in BASELINE_STRATEGIES}
    state["learned_predictions"] = {name: prediction_strategy_valid(name) for name in LEARNED_STRATEGIES}
    state["metrics"] = metrics_valid()
    state["selection"] = selection_valid()
    state["freeze"] = frozen_config_valid()
    return state

def completed_stage_names(state=None):
    state = state or artifact_state()
    completed = []
    if state["pseudo_train"]["full"] and state["mu_train"]["full"]:
        completed.append("train-supervision-complete")
    if completed and state["pseudo_dev"]["full"] and state["mu_dev"]["full"] and all(state["policies"].values()):
        completed.append("dev-supervision-and-policies-complete")
    if len(completed) == 2 and all(state["baseline_predictions"].values()):
        completed.append("dev-baselines-complete")
    if len(completed) == 3 and all(state["learned_predictions"].values()) and state["metrics"] and state["selection"] and state["freeze"]:
        completed.append("dev-frozen-complete")
    return completed

def print_restore_summary(state):
    print("RESTORE SUMMARY")
    print(f"TimelyMT TRAIN pseudo: {state['pseudo_train']['valid']}/{state['pseudo_train']['expected']}")
    print(f"MU TRAIN supervision: {'COMPLETE' if state['mu_train']['full'] else 'MISSING/PARTIAL'}")
    print(f"TimelyMT DEV pseudo: {state['pseudo_dev']['valid']}/{state['pseudo_dev']['expected']}")
    print(f"MU DEV supervision: {'COMPLETE' if state['mu_dev']['full'] else 'MISSING/PARTIAL'}")
    for name in ("P0", "P1", "P2", "mu_zhang2020"):
        print(f"{name.replace('mu_zhang2020', 'MU')}: {'checkpoint valid' if state['policies'][name] else 'checkpoint missing/invalid'}")
    for label, strategies in (("DEV fixed", FIXED_STRATEGIES), ("DEV LA-style", STYLE_STRATEGIES), ("DEV LA-2", ["local_agreement_la2"]), ("DEV MU", ["mu_zhang2020"]), ("DEV learned", LEARNED_STRATEGIES)):
        print(f"{label}: {'COMPLETE' if all(state['baseline_predictions'].get(name, state['learned_predictions'].get(name, False)) for name in strategies) else 'MISSING/PARTIAL'}")
    print(f"DEV evaluation: {'COMPLETE' if state['metrics'] else 'MISSING'}")
    print(f"DEV selection: {'COMPLETE' if state['selection'] else 'MISSING'}")
    print(f"freeze: {'COMPLETE' if state['freeze'] else 'MISSING'}")

def is_test_archive_path(name):
    parts = [part.lower() for part in PurePosixPath(name).parts]
    return len(parts) >= 4 and parts[:3] == ["outputs", "experiments", "research-mvp"] and "test" in parts[3:]

def validated_member_path(member):
    name = member.name
    if not name or "\\" in name or name.startswith("/"):
        raise RuntimeError(f"Unsafe checkpoint archive path: {name!r}")
    path = PurePosixPath(name)
    if path.is_absolute() or any(part in {"", ".", ".."} for part in path.parts):
        raise RuntimeError(f"Unsafe checkpoint archive path: {name!r}")
    if member.issym() or member.islnk() or member.isdev() or not (member.isfile() or member.isdir()):
        raise RuntimeError(f"Unsafe checkpoint archive member type: {name!r}")
    allowed = path == PurePosixPath("checkpoint-metadata.json") or any(path == root or root in path.parents for root in ALLOWED_CHECKPOINT_ROOTS)
    if not allowed:
        raise RuntimeError(f"Unexpected checkpoint archive path: {name!r}")
    if is_test_archive_path(name):
        raise RuntimeError(f"TEST-derived checkpoint artifact is forbidden: {name!r}")
    return path

def validate_archive(archive_path, expected_sha256=None):
    archive_path = Path(archive_path)
    actual_sha256 = sha256_file(archive_path)
    if expected_sha256 and actual_sha256 != expected_sha256:
        raise RuntimeError(f"Checkpoint SHA-256 mismatch: expected {expected_sha256}, got {actual_sha256}")
    with tarfile.open(archive_path, "r:gz") as archive:
        members = archive.getmembers()
        paths = [validated_member_path(member) for member in members]
        if len(paths) != len(set(paths)):
            raise RuntimeError("Checkpoint archive contains duplicate member paths")
        metadata_members = [member for member in members if member.name == "checkpoint-metadata.json" and member.isfile()]
        if len(metadata_members) != 1:
            raise RuntimeError("Checkpoint archive must contain one checkpoint-metadata.json")
        metadata = json.load(archive.extractfile(metadata_members[0]))
    if metadata.get("schema_version") != CHECKPOINT_SCHEMA_VERSION:
        raise RuntimeError(f"Unsupported checkpoint metadata schema: {metadata.get('schema_version')!r}")
    return metadata, actual_sha256, members

def checkpoint_files():
    files = []
    for root in ALLOWED_CHECKPOINT_ROOTS:
        base = repo / Path(*root.parts)
        if not base.exists():
            continue
        for path in sorted(base.rglob("*")):
            relative = path.relative_to(repo)
            if path.is_symlink() or not path.is_file():
                continue
            if any(part in {"__pycache__", "smoke", "tmp", "temp", "cache"} for part in relative.parts):
                continue
            if is_test_archive_path(relative.as_posix()):
                raise RuntimeError(f"TEST-derived artifact blocks checkpoint upload: {relative}")
            files.append((path, relative.as_posix()))
    return files

def checkpoint_metadata(stage_name, completed_stages, files):
    summaries = {}
    for root in ALLOWED_CHECKPOINT_ROOTS:
        prefix = root.as_posix() + "/"
        selected = [path for path, arcname in files if arcname.startswith(prefix)]
        summaries[root.as_posix()] = {"file_count": len(selected), "bytes": sum(path.stat().st_size for path in selected)}
    return {
        "schema_version": CHECKPOINT_SCHEMA_VERSION,
        "created_at_utc": utc_now(),
        "git_commit": commit,
        "dataset_manifest_checksum": DATASET_CHECKSUM,
        "split_checksum": SPLIT_CHECKSUM,
        "translator_config_fingerprint": TRANSLATOR_CONFIG_FINGERPRINT,
        "checkpoint_stage": stage_name,
        "completed_stage_names": list(completed_stages),
        "archive_contents_summary": summaries,
    }

def deterministic_tar_add(archive, source, arcname):
    info = archive.gettarinfo(str(source), arcname=arcname)
    info.uid = info.gid = 0
    info.uname = info.gname = ""
    info.mtime = 0
    info.mode = 0o644
    with source.open("rb") as handle:
        archive.addfile(info, handle)

def build_checkpoint_archive(stage_name, completed_stages):
    state = artifact_state()
    actual_completed = completed_stage_names(state)
    if not set(completed_stages).issubset(actual_completed):
        raise RuntimeError(f"Refusing false completion metadata: requested={completed_stages}, validated={actual_completed}")
    required = CHECKPOINT_STAGE_REQUIREMENTS[stage_name]
    if not required.issubset(actual_completed):
        raise RuntimeError(f"Checkpoint stage {stage_name!r} is incomplete: required={sorted(required)}, validated={actual_completed}")
    files = checkpoint_files()
    if not files:
        raise RuntimeError("No resumable TRAIN/DEV research artifacts exist; refusing empty checkpoint")
    metadata = checkpoint_metadata(stage_name, completed_stages, files)
    metadata_bytes = (json.dumps(metadata, indent=2, sort_keys=True) + "\n").encode("utf-8")
    CHECKPOINT_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    with CHECKPOINT_ARCHIVE.open("wb") as raw:
        with gzip.GzipFile(filename="", mode="wb", fileobj=raw, mtime=0) as compressed:
            with tarfile.open(fileobj=compressed, mode="w", format=tarfile.PAX_FORMAT) as archive:
                info = tarfile.TarInfo("checkpoint-metadata.json")
                info.size, info.mtime, info.mode = len(metadata_bytes), 0, 0o644
                archive.addfile(info, io.BytesIO(metadata_bytes))
                for source, arcname in files:
                    deterministic_tar_add(archive, source, arcname)
    validated_metadata, checksum, members = validate_archive(CHECKPOINT_ARCHIVE)
    if validated_metadata != metadata:
        raise RuntimeError("Checkpoint metadata changed during archive construction")
    print(f"Archive size: {CHECKPOINT_ARCHIVE.stat().st_size} bytes ({CHECKPOINT_ARCHIVE.stat().st_size / 1024**2:.2f} MiB)")
    print(f"Checkpoint stage: {stage_name}")
    print(f"Archive members: {len(members)}")
    print(f"SHA-256: {checksum}")
    return metadata, checksum

def prepare_upload_staging(metadata, checksum):
    if CHECKPOINT_UPLOAD_DIR.exists():
        shutil.rmtree(CHECKPOINT_UPLOAD_DIR)
    CHECKPOINT_UPLOAD_DIR.mkdir(parents=True)
    shutil.copy2(CHECKPOINT_ARCHIVE, CHECKPOINT_UPLOAD_DIR / CHECKPOINT_ARCHIVE.name)
    summary = {"archive": CHECKPOINT_ARCHIVE.name, "archive_sha256": checksum, "archive_size_bytes": CHECKPOINT_ARCHIVE.stat().st_size, **metadata}
    (CHECKPOINT_UPLOAD_DIR / "checkpoint-summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    with tempfile.TemporaryDirectory(prefix="timelymt-kaggle-metadata-") as temporary:
        kaggle_command("datasets", "metadata", CHECKPOINT_DATASET_REF, "-p", temporary)
        candidates = list(Path(temporary).glob("*metadata*.json"))
        if len(candidates) != 1:
            raise RuntimeError(f"Kaggle Dataset metadata download was ambiguous: {candidates}")
        dataset_metadata = json.loads(candidates[0].read_text(encoding="utf-8"))
    dataset_metadata["id"] = CHECKPOINT_DATASET_REF
    (CHECKPOINT_UPLOAD_DIR / "dataset-metadata.json").write_text(json.dumps(dataset_metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    staged = sorted(path.name for path in CHECKPOINT_UPLOAD_DIR.iterdir())
    if staged != ["checkpoint-summary.json", "dataset-metadata.json", "timelymt-checkpoint.tar.gz"]:
        raise RuntimeError(f"Unexpected checkpoint upload staging contents: {staged}")

def checkpoint(stage_name, completed_stages=None):
    if stage_name not in CHECKPOINT_STAGE_REQUIREMENTS:
        raise ValueError(f"Unknown checkpoint stage: {stage_name}")
    completed_stages = completed_stage_names() if completed_stages is None else list(completed_stages)
    metadata, checksum = build_checkpoint_archive(stage_name, completed_stages)
    authenticate_kaggle()
    prepare_upload_staging(metadata, checksum)
    kaggle_command("datasets", "version", "-p", CHECKPOINT_UPLOAD_DIR, "-m", stage_name)
    status = kaggle_command("datasets", "status", CHECKPOINT_DATASET_REF, "--format", "json", capture_output=True).stdout.strip()
    if not status:
        raise RuntimeError("Checkpoint Dataset status verification returned no result")
    print("CHECKPOINT SAVED")
    print(f"stage={stage_name}")
    print(f"sha256={checksum}")
    print(f"dataset={CHECKPOINT_DATASET_REF}")
    return checksum

RESTORED_CHECKPOINT_STAGE = None

def checkpoint_boundary(stage_name):
    restored_rank = CHECKPOINT_STAGE_ORDER.get(RESTORED_CHECKPOINT_STAGE, -1)
    if restored_rank >= CHECKPOINT_STAGE_ORDER[stage_name]:
        print(f"Checkpoint boundary already covered by restored stage {RESTORED_CHECKPOINT_STAGE}; upload skipped")
        return None
    return checkpoint(stage_name, completed_stage_names())

def extract_allowed_artifacts(archive_path):
    with tarfile.open(archive_path, "r:gz") as archive:
        for member in archive.getmembers():
            path = validated_member_path(member)
            if path == PurePosixPath("checkpoint-metadata.json") or member.isdir():
                continue
            target = (repo / Path(*path.parts)).resolve()
            if not target.is_relative_to(repo.resolve()):
                raise RuntimeError(f"Checkpoint extraction escaped repository: {target}")
            target.parent.mkdir(parents=True, exist_ok=True)
            source = archive.extractfile(member)
            if source is None:
                raise RuntimeError(f"Checkpoint member is unreadable: {member.name}")
            with target.open("wb") as handle:
                shutil.copyfileobj(source, handle)

def restore_latest_checkpoint():
    global RESTORED_CHECKPOINT_STAGE
    with tempfile.TemporaryDirectory(prefix="timelymt-checkpoint-restore-") as temporary:
        download_dir = Path(temporary)
        try:
            kaggle_command("datasets", "download", CHECKPOINT_DATASET_REF, "-p", download_dir, "--unzip")
        except subprocess.CalledProcessError as error:
            print(f"NO COMPATIBLE CHECKPOINT — STARTING CLEAN ({type(error).__name__})")
            return None
        archives = list(download_dir.rglob("timelymt-checkpoint.tar.gz"))
        summaries = list(download_dir.rglob("checkpoint-summary.json"))
        if len(archives) != 1:
            print("NO COMPATIBLE CHECKPOINT — STARTING CLEAN")
            return None
        summary = load_json_if_valid(summaries[0]) if len(summaries) == 1 else None
        expected_sha = summary.get("archive_sha256") if summary else None
        try:
            metadata, _, _ = validate_archive(archives[0], expected_sha)
            compatible = (
                metadata.get("git_commit") == commit
                and metadata.get("dataset_manifest_checksum") == DATASET_CHECKSUM
                and metadata.get("split_checksum") == SPLIT_CHECKSUM
                and metadata.get("translator_config_fingerprint") == TRANSLATOR_CONFIG_FINGERPRINT
                and metadata.get("checkpoint_stage") in CHECKPOINT_STAGE_ORDER
            )
            if not compatible:
                raise RuntimeError("Checkpoint research identity is incompatible with this clone")
            extract_allowed_artifacts(archives[0])
        except (OSError, tarfile.TarError, RuntimeError, json.JSONDecodeError) as error:
            print(f"NO COMPATIBLE CHECKPOINT — STARTING CLEAN ({error})")
            return None
    RESTORED_CHECKPOINT_STAGE = metadata["checkpoint_stage"]
    print(f"RESTORED CHECKPOINT: {RESTORED_CHECKPOINT_STAGE}")
    print_restore_summary(artifact_state())
    return metadata

## RESTORE CHECKPOINT
Download and safely restore the latest compatible private Dataset version before expensive EnViT5 computation. Source, configs, Dataset v1, schemas, tests, and notebooks can never be archive targets.

In [ ]:
kaggle_dataset_status = authenticate_kaggle()
restored_checkpoint_metadata = restore_latest_checkpoint()

## MODEL/CACHE SETUP
Download the frozen EnViT5 revision into Kaggle temporary storage, never into Git. The experiment does not fine-tune EnViT5.

In [ ]:
if not (package_preflight_passed and environment_preflight_passed and dataset_preflight_passed and split_identity_passed and cuda_preflight_passed):
    raise RuntimeError("Package, environment, Dataset v1, split, and CUDA preflights must pass before model setup")

MODEL_ID = "VietAI/envit5-translation"
MODEL_REVISION = "840bc88104d5a4277af740eaedb024df8c3093e7"

from huggingface_hub import snapshot_download
from timelymt.translator.envit5 import load_config

translator_config = load_config(repo / "configs/translator/envit5.json")
if not (
    translator_config.frozen
    and translator_config.model_id == MODEL_ID
    and translator_config.model_revision == MODEL_REVISION
):
    raise RuntimeError("Translator config is not the expected frozen EnViT5 revision")
snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION, cache_dir=os.environ["HF_HUB_CACHE"])
print(f"Pinned model cached under: {HF_CACHE_DIR}")

## MODEL PREFLIGHT
Run one tiny real translation through the existing TimelyMT translator API. This verifies the pinned revision, CUDA float16 inference, and removal of EnViT5's leading `vi:` control tag.

In [ ]:
if not (dataset_preflight_passed and cuda_preflight_passed):
    raise RuntimeError("Dataset v1 and CUDA preflights must pass before model inference")

from timelymt.translator.envit5 import EnViT5Translator

smoke_translator = EnViT5Translator(translator_config)
smoke_result = smoke_translator.translate("Artificial intelligence helps people")
runtime_info = smoke_translator.runtime_info()
if runtime_info["model_revision"] != MODEL_REVISION:
    raise RuntimeError(f"Loaded unexpected model revision: {runtime_info['model_revision']}")
if runtime_info["device"] != "cuda" or runtime_info["dtype"] != "float16":
    raise RuntimeError(f"Unexpected inference runtime: {runtime_info}")
if smoke_result.translated_text.startswith("vi:"):
    raise RuntimeError("Normalized EnViT5 output still contains the leading vi: control tag")
print(json.dumps({**runtime_info, "normalized_output": smoke_result.translated_text}, indent=2, ensure_ascii=False))
model_preflight_passed = True
del smoke_translator
torch.cuda.empty_cache()

## POST-RESTORE VALIDATION
Derive resumability from restored artifacts. Existing TimelyMT supervision validators remain authoritative; invalid or partial stages are not promoted to complete.

In [ ]:
if restored_checkpoint_metadata is not None:
    for validator_stage, split_name, key in (
        ("validate-pseudo", "train", "pseudo_train"),
        ("validate-mu", "train", "mu_train"),
        ("validate-pseudo", "dev", "pseudo_dev"),
        ("validate-mu", "dev", "mu_dev"),
    ):
        current = artifact_state()[key]
        if current["valid"]:
            cli(validator_stage, "--split", split_name)
    RESTORE_STATE = artifact_state()
    print_restore_summary(RESTORE_STATE)
else:
    RESTORE_STATE = artifact_state()

## EMERGENCY CHECKPOINT
Set `RUN_EMERGENCY_CHECKPOINT = True` and run the next cell to persist all currently valid resumable artifacts. It performs no experiment.

In [ ]:
# EMERGENCY CHECKPOINT
if RUN_EMERGENCY_CHECKPOINT:
    checkpoint(stage_name="manual-emergency", completed_stages=completed_stage_names())
else:
    print("Emergency checkpoint armed only when RUN_EMERGENCY_CHECKPOINT=True")

## KAGGLE PRECHECK CHECKPOINT
Summarize every prerequisite immediately before the unchanged FULL TRAIN + DEV pipeline. `READY` is unreachable unless all bootstrap, Dataset v1, CUDA, frozen-model, and TEST-safeguard gates passed.

In [ ]:
precheck_results = [
    ("Git clone", git_clone_passed, ""),
    ("Source-tree import", source_tree_import_passed, ""),
    ("Project root", project_root_passed, ""),
    ("TimelyMT import", package_preflight_passed, ""),
    ("Dataset v1", dataset_preflight_passed, f" ({len(canonical_paths)}/17)"),
    ("Split identity", split_identity_passed, ""),
    ("CUDA", cuda_preflight_passed, ""),
    ("Frozen EnViT5", model_preflight_passed, ""),
    ("TEST safeguard", test_safeguard_passed, ""),
]
print("KAGGLE PRECHECK")
print("---------------")
for label, passed, detail in precheck_results:
    print(f"{label:<23} {'PASS' if passed else 'FAIL'}{detail}")
if not all(passed for _, passed, _ in precheck_results):
    raise RuntimeError("Kaggle precheck failed; FULL TRAIN + DEV is blocked")
print("\nREADY FOR FULL TRAIN + DEV")

## FULL TRAIN — TIMELYMT PSEUDO LABELS
Generate full TRAIN future-stability supervision on GPU. A fresh public clone starts at 0/12 talks; rerunning after interruption reuses valid completed talk-level artifacts.

If CUDA OOM occurs, change only `INFERENCE_BATCH_SIZE`: 3 -> 2 -> 1.
<!-- Compatibility heading: ## TIMELYMT TRAIN PSEUDO -->

In [ ]:
elapsed_session()
if supervision_status("pseudo", "train")["full"]:
    print("FULL TRAIN TimelyMT pseudo labels validated; generation skipped")
else:
    cli("pseudo", "--split", "train", "--batch-size", str(INFERENCE_BATCH_SIZE))

## FULL TRAIN — ZHANG 2020 MU SUPERVISION
Generate full TRAIN oracle supervision for the frozen MU literature adaptation on GPU. This is TRAIN-only supervision; MU runtime remains causal.

In [ ]:
if supervision_status("mu", "train")["full"]:
    print("FULL TRAIN MU supervision validated; generation skipped")
else:
    cli("mu-supervision", "--split", "train", "--batch-size", str(INFERENCE_BATCH_SIZE))

## CHECKPOINT A — TRAIN SUPERVISION COMPLETE
Persist full TRAIN TimelyMT and MU supervision as a new private Dataset version.

In [ ]:
for validator_stage in ("validate-pseudo", "validate-mu"):
    cli(validator_stage, "--split", "train")
checkpoint_boundary("train-supervision-complete")

## FULL DEV — TIMELYMT PSEUDO LABELS
Generate full DEV supervision on GPU for freeze prerequisites and provenance. Valid talk-level artifacts are resumable.
<!-- Compatibility heading: ## TIMELYMT DEV PSEUDO -->

In [ ]:
elapsed_session()
if supervision_status("pseudo", "dev")["full"]:
    print("FULL DEV TimelyMT pseudo labels validated; generation skipped")
else:
    cli("pseudo", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE))

## FULL DEV — ZHANG 2020 MU SUPERVISION
Generate full DEV MU supervision on GPU for freeze prerequisites.
<!-- Compatibility heading: ## MU TRAIN/DEV SUPERVISION -->

In [ ]:
if supervision_status("mu", "dev")["full"]:
    print("FULL DEV MU supervision validated; generation skipped")
else:
    cli("mu-supervision", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE))

## VALIDATE SUPERVISION
Require full, publishable TRAIN/DEV manifests and exact split coverage before classifier training.
<!-- Compatibility heading: ## VALIDATE -->

In [ ]:
for stage, split_name in (
    ("validate-pseudo", "train"),
    ("validate-pseudo", "dev"),
    ("validate-mu", "train"),
    ("validate-mu", "dev"),
):
    cli(stage, "--split", split_name)
for relative in (
    "data/policy/pseudo_labels/train/manifest.json",
    "data/policy/pseudo_labels/dev/manifest.json",
    "data/policy/mu_zhang2020/train/manifest.json",
    "data/policy/mu_zhang2020/dev/manifest.json",
):
    document = json.loads(require_file(relative).read_text(encoding="utf-8"))
    if document.get("artifact_status") != "full" or document.get("publishable") is not True:
        raise RuntimeError(f"Supervision is not full/publishable: {relative}")

## TRAIN P0
Train the lightweight P0 policy classifier on CPU. EnViT5 remains frozen.
<!-- Compatibility heading: ## TRAIN P0/P1/P2 -->

In [ ]:
if policy_checkpoint_valid("P0"):
    print("P0 checkpoint validated; training skipped")
else:
    cli("train", "--pseudo-labels", "data/policy/pseudo_labels/train/manifest.json", "--variant", "P0")
if not policy_checkpoint_valid("P0"):
    raise RuntimeError("P0 checkpoint failed post-training validation")

## TRAIN P1
Train the lightweight P1 policy classifier on CPU.

In [ ]:
if policy_checkpoint_valid("P1"):
    print("P1 checkpoint validated; training skipped")
else:
    cli("train", "--pseudo-labels", "data/policy/pseudo_labels/train/manifest.json", "--variant", "P1")
if not policy_checkpoint_valid("P1"):
    raise RuntimeError("P1 checkpoint failed post-training validation")

## TRAIN P2
Train the lightweight P2 classifier. P2 history remains system-generated target history only.

In [ ]:
if policy_checkpoint_valid("P2"):
    print("P2 checkpoint validated; training skipped")
else:
    cli("train", "--pseudo-labels", "data/policy/pseudo_labels/train/manifest.json", "--variant", "P2")
if not policy_checkpoint_valid("P2"):
    raise RuntimeError("P2 checkpoint failed post-training validation")

## TRAIN MU
Train the lightweight Zhang-2020 MU adaptation classifier on CPU.

In [ ]:
if policy_checkpoint_valid("mu_zhang2020"):
    print("MU checkpoint validated; training skipped")
else:
    cli("train-mu", "--pseudo-labels", "data/policy/mu_zhang2020/train/manifest.json")
if not policy_checkpoint_valid("mu_zhang2020"):
    raise RuntimeError("MU checkpoint failed post-training validation")

## CHECKPOINT B — DEV SUPERVISION AND POLICIES COMPLETE
Persist full TRAIN/DEV supervision and validated P0/P1/P2/MU checkpoints.

In [ ]:
checkpoint_boundary("dev-supervision-and-policies-complete")

## DEV FIXED BASELINES
Run all frozen Fixed-N and Fixed-Time DEV baselines on GPU. Completed prediction files are resumable.
<!-- Compatibility heading: ## DEV BASELINES -->

In [ ]:
FIXED = FIXED_STRATEGIES
elapsed_session()
if all(prediction_strategy_valid(strategy) for strategy in FIXED):
    print("All fixed DEV predictions validated; rollout skipped")
else:
    cli("rollout", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE), "--strategies", *FIXED)

## DEV LOCAL AGREEMENT STYLE
Run TimelyMT's frozen `local_agreement_style` k=2 and k=3 heuristic baselines on GPU.

In [ ]:
STYLE = STYLE_STRATEGIES
if all(prediction_strategy_valid(strategy) for strategy in STYLE):
    print("All local-agreement-style DEV predictions validated; rollout skipped")
else:
    cli("rollout", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE), "--strategies", *STYLE)

## DEV LOCAL AGREEMENT LA-2
Run the frozen literature LA-2 adaptation on DEV using GPU inference.
<!-- Compatibility heading: ## DEV LA-2 -->

In [ ]:
if prediction_strategy_valid("local_agreement_la2"):
    print("LA-2 DEV predictions validated; rollout skipped")
else:
    cli("rollout", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE), "--strategies", "local_agreement_la2")

## DEV MU
Run the trained Zhang-2020 MU adaptation on DEV using GPU inference.
<!-- Compatibility heading: ## DEV MU ROLLOUT -->

In [ ]:
if prediction_strategy_valid("mu_zhang2020"):
    print("MU DEV predictions validated; rollout skipped")
else:
    cli("rollout", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE), "--strategies", "mu_zhang2020")

## CHECKPOINT C — DEV BASELINES COMPLETE
Critical persistence boundary after fixed baselines, local-agreement style, LA-2, and Zhang-2020 MU DEV rollout.

In [ ]:
if not all(prediction_strategy_valid(strategy) for strategy in BASELINE_STRATEGIES):
    raise RuntimeError("DEV baseline checkpoint boundary has incomplete or invalid predictions")
checkpoint_boundary("dev-baselines-complete")

## DEV TIMELYMT POLICIES
Run P0/P1/P2 causally across every preregistered probability threshold on DEV.
<!-- Compatibility heading: ## DEV LEARNED ROLLOUT -->

In [ ]:
LEARNED = LEARNED_STRATEGIES
elapsed_session()
if all(prediction_strategy_valid(strategy) for strategy in LEARNED):
    print("All learned-policy DEV predictions validated; rollout skipped")
else:
    cli("rollout", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE), "--strategies", *LEARNED)

## DEV EVALUATION
Compute frozen SacreBLEU, chrF2, AL, LAAL, and supporting latency/commit statistics from complete DEV predictions.
<!-- Compatibility heading: ## DEV EVALUATE -->

In [ ]:
BASELINES = BASELINE_STRATEGIES
ALL_STRATEGIES = BASELINES + LEARNED
if metrics_valid():
    print("Complete DEV metrics validated; evaluation skipped")
else:
    cli("evaluate", "--split", "dev", "--strategies", *ALL_STRATEGIES)
metrics = json.loads(require_file("outputs/experiments/research-mvp/metrics/dev/all.json").read_text(encoding="utf-8"))
if not metrics_valid():
    raise RuntimeError("DEV metrics are incomplete or not full")

## DEV SELECTION
Apply the frozen deterministic TimelyMT-only P0/P1/P2 selection rule. MU and LA-2 remain comparison baselines and cannot alter selection.
<!-- Compatibility heading: ## DEV SELECT -->

In [ ]:
if selection_valid():
    print("DEV selection validated; selection skipped")
else:
    cli("select")
selection = json.loads(require_file("outputs/experiments/research-mvp/dev-selection.json").read_text(encoding="utf-8"))
if not selection_valid():
    raise RuntimeError(f"Invalid TimelyMT DEV selection: {selection}")

## FREEZE FINAL EVALUATION CONFIG
Create the immutable evaluation configuration only after all full supervision, checkpoints, DEV metrics, and DEV selection pass their gates.
<!-- Compatibility heading: ## FREEZE -->

In [ ]:
import hashlib

if frozen_config_valid():
    print("Frozen evaluation config validated; freeze skipped")
else:
    cli("freeze")
frozen_path = require_file("outputs/experiments/research-mvp/frozen-eval-config.json")
frozen = json.loads(frozen_path.read_text(encoding="utf-8"))
if not frozen_config_valid():
    raise RuntimeError("Frozen evaluation config failed identity validation")
print(f"Frozen config: {frozen_path}")
print(f"SHA-256: {hashlib.sha256(frozen_path.read_bytes()).hexdigest()}")

## EXPORT ARTIFACTS
Package generated TRAIN/DEV supervision, trained policy checkpoints, DEV predictions/metrics/selection, frozen config, and compact provenance. Canonical Dataset v1 and all model/translator caches are excluded.

In [ ]:
import platform
import tarfile

experiment_dir = repo / "outputs/experiments/research-mvp"
provenance_path = experiment_dir / "kaggle-run-provenance.json"
provenance_path.write_text(json.dumps({
    "repository_url": REPO_URL,
    "repository_branch": REPO_BRANCH,
    "repository_commit": commit,
    "dataset_checksum": DATASET_CHECKSUM,
    "split_checksum": SPLIT_CHECKSUM,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "inference_batch_size": INFERENCE_BATCH_SIZE,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "gpu_name": torch.cuda.get_device_name(0),
}, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")

EXPORT = Path("/kaggle/working/timelymt-research-mvp-artifacts.tar.gz")
ARTIFACT_DIRS = [
    Path("data/policy/pseudo_labels/train"),
    Path("data/policy/pseudo_labels/dev"),
    Path("data/policy/mu_zhang2020/train"),
    Path("data/policy/mu_zhang2020/dev"),
    Path("outputs/experiments/research-mvp"),
]
ARTIFACT_FILES = [
    *(Path("checkpoints/policy") / name for name in (
        "P0.joblib", "P0.metadata.json", "P1.joblib", "P1.metadata.json",
        "P2.joblib", "P2.metadata.json",
        "mu_zhang2020.joblib", "mu_zhang2020.metadata.json",
    )),
    Path("data/manifests/streaming-dataset.json"),
    Path("data/manifests/timelymt-streaming-dataset-v1.json"),
    Path("data/splits/experimental.json"),
    Path("configs/translator/envit5.json"),
    Path("configs/experiments/research-mvp.json"),
]
for relative in ARTIFACT_DIRS:
    if not (repo / relative).is_dir():
        raise RuntimeError(f"Missing required artifact directory: {repo / relative}")
for relative in ARTIFACT_FILES:
    require_file(relative)

def archive_filter(info):
    parts = Path(info.name).parts
    forbidden = {".git", "__pycache__", "cache"}
    return None if forbidden.intersection(parts) else info

with tarfile.open(EXPORT, "w:gz") as archive:
    for relative in ARTIFACT_DIRS:
        archive.add(repo / relative, arcname=relative.as_posix(), filter=archive_filter)
    for relative in ARTIFACT_FILES:
        archive.add(repo / relative, arcname=relative.as_posix(), filter=archive_filter)

with tarfile.open(EXPORT, "r:gz") as archive:
    top_levels = sorted({Path(member.name).parts[0] for member in archive.getmembers() if member.name})
print(f"Archive: {EXPORT}")
print(f"Archive size: {EXPORT.stat().st_size} bytes ({EXPORT.stat().st_size / 1024**2:.2f} MiB)")
print(f"Top-level archived directories: {top_levels}")
checkpoint_boundary("dev-frozen-complete")

# STOP BEFORE TEST
TRAIN is complete, DEV selection is complete, and the experiment configuration is frozen. Execute the held-out evaluation separately using this frozen configuration. Do not use held-out data to tune thresholds, features, policies, baselines, or any other research choice. Kaggle exposes `/kaggle/working/timelymt-research-mvp-artifacts.tar.gz` as the downloadable notebook output.